# Merging, Joining, and Concatenating

---

### Table of Contents
1. Introduction and Setup
2. Concatenating DataFrames (`pd.concat`)
3. Merging DataFrames (`pd.merge`)
4. Joining DataFrames (`.join`)

---

## 1. Introduction and Setup
- Data often comes from multiple sources and needs to be combined.
- Pandas provides three primary ways to combine DataFrames:
  1. `pd.concat()`: Stacking/appending DataFrames.
  2. `pd.merge()`: SQL-style joins on common columns.
  3. `.join()`: A convenient method for index-based joins.

In [1]:
# --- Setup: Create sample DataFrames for our examples ---

import pandas as pd

# Employee data
staff_df = pd.DataFrame(
    {
        "employee_id": ["E1", "E2", "E3", "E4"],
        "name": ["Alice", "Bob", "Charlie", "David"],
        "department_id": ["D1", "D2", "D1", "D3"],
    }
)

# Department data
dept_df = pd.DataFrame(
    {
        "department_id": ["D1", "D2", "D4"],
        "department_name": ["Sales", "Engineering", "Marketing"],
    }
)

# Salary data
salary_df = pd.DataFrame(
    {"employee_id": ["E1", "E2", "E3", "E4"], "salary": [70000, 80000, 65000, 90000]}
)

print("--- Sample DataFrames ---")
print("Staff DataFrame:\n", staff_df)
print("\nDepartment DataFrame:\n", dept_df)
print("\nSalary DataFrame:\n", salary_df)

--- Sample DataFrames ---
Staff DataFrame:
   employee_id     name department_id
0          E1    Alice            D1
1          E2      Bob            D2
2          E3  Charlie            D1
3          E4    David            D3

Department DataFrame:
   department_id department_name
0            D1           Sales
1            D2     Engineering
2            D4       Marketing

Salary DataFrame:
   employee_id  salary
0          E1   70000
1          E2   80000
2          E3   65000
3          E4   90000



---

## 2. Concatenating DataFrames (`pd.concat`)
- Concatenation "stacks" DataFrames on top of each other (axis=0)
  or side-by-side (axis=1). It's primarily for combining data with the same structure.

In [2]:
# --- Vertical Concatenation (axis=0) ---

new_staff_df = pd.DataFrame(
    {"employee_id": ["E5"], "name": ["Eve"], "department_id": ["D2"]}
)
all_staff = pd.concat([staff_df, new_staff_df], ignore_index=True)
# `ignore_index=True` creates a new, clean index for the combined DataFrame.
print("Vertically concatenated staff data:\n", all_staff)

Vertically concatenated staff data:
   employee_id     name department_id
0          E1    Alice            D1
1          E2      Bob            D2
2          E3  Charlie            D1
3          E4    David            D3
4          E5      Eve            D2



---

## 3. Merging DataFrames (`pd.merge`)
- `pd.merge()` is the primary tool for combining DataFrames based on the
  values in common columns, like a JOIN in SQL.

In [3]:
# --- Inner Join (Default) ---

# - Keeps only the rows where the key ('department_id') exists in BOTH DataFrames.
# - Notice 'David' (D3) and 'Marketing' (D4) are dropped.
inner_join = pd.merge(staff_df, dept_df, on="department_id", how="inner")
print("Inner Join:\n", inner_join)

Inner Join:
   employee_id     name department_id department_name
0          E1    Alice            D1           Sales
1          E2      Bob            D2     Engineering
2          E3  Charlie            D1           Sales


In [4]:
# --- Outer Join ---

# - Keeps ALL rows from both DataFrames, filling with `NaN` where data is missing.
# - Notice 'David' and 'Marketing' are now included.
outer_join = pd.merge(staff_df, dept_df, on="department_id", how="outer")
print("\nOuter Join:\n", outer_join)



Outer Join:
   employee_id     name department_id department_name
0          E1    Alice            D1           Sales
1          E3  Charlie            D1           Sales
2          E2      Bob            D2     Engineering
3          E4    David            D3             NaN
4         NaN      NaN            D4       Marketing


In [5]:
# --- Left Join ---

# - Keeps ALL rows from the LEFT DataFrame (`staff_df`) and only matching rows from the right.
# - 'David' is kept, but his department_name is NaN. 'Marketing' is dropped.
left_join = pd.merge(staff_df, dept_df, on="department_id", how="left")
print("\nLeft Join:\n", left_join)


Left Join:
   employee_id     name department_id department_name
0          E1    Alice            D1           Sales
1          E2      Bob            D2     Engineering
2          E3  Charlie            D1           Sales
3          E4    David            D3             NaN


In [6]:
# --- Right Join ---

# - Keeps ALL rows from the RIGHT DataFrame (`dept_df`) and only matching rows from the left.
# - 'Marketing' is kept. 'David' is dropped.
right_join = pd.merge(staff_df, dept_df, on="department_id", how="right")
print("\nRight Join:\n", right_join)


Right Join:
   employee_id     name department_id department_name
0          E1    Alice            D1           Sales
1          E3  Charlie            D1           Sales
2          E2      Bob            D2     Engineering
3         NaN      NaN            D4       Marketing



---

## 4. Joining DataFrames (`.join`)
- `.join()` is a convenient method for merging that works on the DataFrames' indices.
- It performs a left join by default.
- It's a quick alternative to `merge` when the join key is the index.

In [8]:
# Let's set the index to the column we want to join on
staff_indexed = staff_df.set_index("employee_id")
salary_indexed = salary_df.set_index("employee_id")

print("Staff DataFrame (indexed):\n", staff_indexed)
print("\nSalary DataFrame (indexed):\n", salary_indexed)

Staff DataFrame (indexed):
                 name department_id
employee_id                       
E1             Alice            D1
E2               Bob            D2
E3           Charlie            D1
E4             David            D3

Salary DataFrame (indexed):
              salary
employee_id        
E1            70000
E2            80000
E3            65000
E4            90000


In [9]:
# Join the two DataFrames on their common index
staff_with_salary = staff_indexed.join(salary_indexed)
print("\nJoined staff and salary data:\n", staff_with_salary)


Joined staff and salary data:
                 name department_id  salary
employee_id                               
E1             Alice            D1   70000
E2               Bob            D2   80000
E3           Charlie            D1   65000
E4             David            D3   90000


*Note: `pd.merge` can also join on indices using `left_index=True` and `right_index=True`. `merge` is the more powerful and flexible function overall.*

---

**Next:** [Time Series Analysis](./11_time_series_analysis.ipynb)